# Reproducing Manuscript Tables and Figures (Tier 1)

This notebook reproduces all tables, statistical tests, sensitivity audits, and publication figures for:

> **Global flood models can close nearly two-thirds of the warning gap**

All computations execute deterministically on CPU from the companion Zenodo derived benchmark archive (`10.5281/zenodo.23805977` — *placeholder DOI; finalized upon archive publication*). Zero proprietary credentials, raw CAP alert texts, or third-party raw geometry archives are required.

### Prerequisites
1. Install the package with notebook support: `pip install -e "..[notebooks]"` (or `pip install -e ".[notebooks]"` from the repository root).
2. Download and unpack the companion Zenodo archive, then set `CAP_GAP_ZENODO_DIR` in your environment (or edit `ZENODO_DIR` below).

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

from cap_alerting_coverage.cli import (
    PARITY_TABLES,
    PASS_THROUGH_TAGS,
    VALIDATION_PARITY_TABLES,
    compare_csv_tables,
)
from cap_alerting_coverage.config import ProjectPaths
from cap_alerting_coverage.manuscript_outputs import generate_manuscript_tables_and_figures
from cap_alerting_coverage.schemas import (
    validate_cap_gid0_matrix,
    validate_cap_gid1_matrix,
    validate_deidentified_validation_table,
    validate_event_catalog,
    validate_ffm_gid1_matrix,
    validate_tier_cube,
)
from cap_alerting_coverage.scoring import run_full_scoring
from cap_alerting_coverage.validation_audit import (
    compute_confusion_matrix_table,
    compute_spatial_precision_table,
    load_deidentified_validation_table,
)
from cap_alerting_coverage.zenodo_fetch import verify_zenodo_checksums

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
ZENODO_DIR = Path(os.environ.get("CAP_GAP_ZENODO_DIR", str(REPO_ROOT / "zenodo_archive"))).expanduser().resolve()
RESULTS_DIR = Path(os.environ.get("CAP_GAP_RESULTS_DIR", str(REPO_ROOT / "out_results"))).expanduser().resolve()
FIGURES_DIR = Path(os.environ.get("CAP_GAP_FIGURES_DIR", str(REPO_ROOT / "out_figures"))).expanduser().resolve()

paths = ProjectPaths.from_env(
    repo_root=REPO_ROOT,
    zenodo_dir=ZENODO_DIR,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
)
print("Configured ProjectPaths successfully (Zenodo dir exists:", paths.zenodo_dir.is_dir(), ")")

Configured ProjectPaths successfully (Zenodo dir exists: True )


## 1. Verify Zenodo Archive Checksums & Schema / Privacy Contracts

Check `SHA256SUMS.txt` and verify that every derived input catalog, Admin-1 exceedance matrix, discrete return-period tier cube, and de-identified `N = 400` validation file satisfies strict schema and non-redistribution guardrails.

In [2]:
if (paths.zenodo_dir / "SHA256SUMS.txt").exists():
    verified = verify_zenodo_checksums(paths.zenodo_dir)
    print(f"SHA-256 verified {len(verified)} files in staged Zenodo archive.")
else:
    print("SHA256SUMS.txt not found; skipping checksum verification.")

validate_event_catalog(
    pd.read_parquet(paths.resolve_data_file("catalog_without_gs_all_185_2021_2025.parquet", "01_event_catalogs")),
    dataset_name="Catalog A (185)",
)
validate_event_catalog(
    pd.read_parquet(paths.resolve_data_file("catalog_with_gs_all_185_2021_2025.parquet", "01_event_catalogs")),
    dataset_name="Catalog B (185)",
)
validate_cap_gid1_matrix(
    pd.read_parquet(paths.resolve_data_file("cap_gid1_daily_2020_2025.parquet", "02_admin1_exceedance_matrices"))
)
validate_cap_gid0_matrix(
    pd.read_parquet(paths.resolve_data_file("cap_gid0_daily_2020_2025.parquet", "02_admin1_exceedance_matrices"))
)
validate_ffm_gid1_matrix(
    pd.read_parquet(paths.resolve_data_file("ffm_gid1_daily_2021_2025.parquet", "02_admin1_exceedance_matrices"))
)

dhm_npz = dict(np.load(paths.resolve_data_file("dhm_10d_gid1_multilead_2021_2025.npz", "02_admin1_exceedance_matrices"), allow_pickle=True))
validate_tier_cube(dhm_npz, expected_ndim=3, gid1_list=dhm_npz["gid_1"], dataset_name="DHM_10D")
print(f"All derived schemas & privacy checks passed ({len(dhm_npz['gid_1']):,} Admin-1 units).")

SHA-256 verified 50 files in staged Zenodo archive.


All derived schemas & privacy checks passed (3,664 Admin-1 units).


## 2. Run Full Tier-1 Benchmark Scoring, Tables, and Figures

In [3]:
scoring_state = run_full_scoring(paths, strict_185=True)
outputs = generate_manuscript_tables_and_figures(
    paths,
    scoring_state=scoring_state,
    strict_185=True,
    render_figures=True,
)
print(f"Generated {len(outputs['figures'])} publication figures (300 DPI).")

Loaded all Admin-1 exceedance cubes & catalogs into RAM in 1.4s.


Manuscript tables & 13 figures generated in 21.3s.


Generated 13 publication figures (300 DPI).


ref_dir = paths.zenodo_dir / "03_summary_results"
val_ref_dir = paths.zenodo_dir / "04_validation_deidentified"
if ref_dir.is_dir():
    rows = []
    for fname in PARITY_TABLES:
        ref_p = ref_dir / fname
        act_p = paths.results_dir / fname
        ok, msg = compare_csv_tables(act_p, ref_p, atol=1e-3)
        tag = PASS_THROUGH_TAGS.get(fname, "").strip(" []") or "RECOMPUTED"
        rows.append({"table": fname, "mode": tag, "status": "PASS" if ok else "FAIL", "detail": msg})
    if val_ref_dir.is_dir():
        for fname in VALIDATION_PARITY_TABLES:
            ref_p = val_ref_dir / fname
            act_p = paths.results_dir / fname
            if ref_p.exists() and act_p.exists():
                ok, msg = compare_csv_tables(act_p, ref_p, atol=1e-3)
                rows.append({"table": fname, "mode": "VALIDATION", "status": "PASS" if ok else "FAIL", "detail": msg})
    parity_df = pd.DataFrame(rows)
    display(parity_df)
    assert (parity_df["status"] == "PASS").all(), "One or more parity checks failed!"
else:
    print("Reference 03_summary_results/ directory not present; skipping parity diff.")

In [4]:
ref_dir = paths.zenodo_dir / "03_summary_results"
if ref_dir.is_dir():
    rows = []
    for fname in PARITY_TABLES:
        ref_p = ref_dir / fname
        act_p = paths.results_dir / fname
        if ref_p.exists() and act_p.exists():
            ok, msg = compare_csv_tables(act_p, ref_p, atol=1e-3)
            rows.append({"table": fname, "status": "PASS" if ok else "FAIL", "detail": msg})
    parity_df = pd.DataFrame(rows)
    display(parity_df)
    assert (parity_df["status"] == "PASS").all(), "One or more parity checks failed!"
else:
    print("Reference 03_summary_results/ directory not present; skipping parity diff.")

,table,status,detail
0,table1_summary.csv,PASS,"PASS (64 rows, max |diff|=0.00e+00)"
1,table2_gauge_vs_catalogs.csv,PASS,"PASS (7 rows, max |diff|=0.00e+00)"
2,stream_gauge_nse_summary.csv,PASS,"PASS (1 rows, max |diff|=0.00e+00)"
3,ed_table1_cohort_d_35.csv,PASS,"PASS (35 rows, max |diff|=0.00e+00)"
4,ed_table3_sensitivity_matrix.csv,PASS,"PASS (174 rows, max |diff|=0.00e+00)"
5,ed_table5_full_benchmark.csv,PASS,"PASS (1039 rows, max |diff|=0.00e+00)"
6,ed_table6a_annual_onboarding.csv,PASS,"PASS (5 rows, max |diff|=0.00e+00)"
7,ed_table6b_late_onboarders.csv,PASS,"PASS (25 rows, max |diff|=0.00e+00)"
8,country_coverage_and_gains_150.csv,PASS,"PASS (185 rows, max |diff|=0.00e+00)"
9,pr_curves_summary.csv,PASS,"PASS (2125 rows, max |diff|=0.00e+00)"


## 4. Inspect Manuscript Tables (Tables 1–2 & Extended Data Tables 1, 3–6)

In [5]:
print("=== Table 1: Headline Coverage & Hybrid Gap Closure ===")
display(pd.read_csv(paths.results_dir / "table1_summary.csv"))

if (paths.results_dir / "table2_gauge_vs_catalogs.csv").exists():
    print("=== Table 2: Stream-Gauge Exceedance Precision vs. Disaster Catalogs ===")
    display(pd.read_csv(paths.results_dir / "table2_gauge_vs_catalogs.csv"))

print("=== Extended Data Table 3: Sensitivity Audit Matrix ===")
display(pd.read_csv(paths.results_dir / "ed_table3_sensitivity_matrix.csv"))

print("=== Extended Data Table 6a: Annual CAP Onboarding Trajectory (2021-2025) ===")
display(pd.read_csv(paths.results_dir / "ed_table6a_annual_onboarding.csv"))

=== Table 1: Headline Coverage & Hybrid Gap Closure ===


,catalog,benchmark,system,label,thr_label,n_events,tp_ev,tp_pre,recall_pct,pre_recall_pct,pop_recall_pct,n_episodes,tp_ep,precision_pct,f1_pct,net_gain_over_cap,pp_gain_over_cap,gap_closed_pct,mcnemar_p
0,Without_GS,2021_2025,CAP_Perm,Official CAP Baseline (Broad Flood-and-Storm),All,5894,2277,2147,38.633,36.427,59.398,65874,2509,3.809,6.934,NaN,NaN,NaN,NaN
1,Without_GS,2021_2025,CAP_Rest,Official CAP Explicit Flood-Coded Only,All,5894,423,368,7.177,6.244,28.938,12903,487,3.774,4.947,NaN,NaN,NaN,NaN
2,Without_GS,2021_2025,FFM,"Google FFM (0–24h, τ=0.50)",tau=0.50,5894,3835,3597,65.066,61.028,79.906,75920,3620,4.768,8.885,NaN,NaN,NaN,NaN
3,Without_GS,2021_2025,DHM_0d,"Google DHM (0d Nowcast, Q2yr)",RP=2,5894,2368,1910,40.176,32.406,73.746,25898,2302,8.889,14.557,NaN,NaN,NaN,NaN
4,Without_GS,2021_2025,DHM_Oper,Google DHM Operational (0–7d valid-date aligne...,RP=2,5894,2588,2144,43.909,36.376,75.946,30202,2424,8.026,13.571,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,With_GS,2024_2025,CAP_Perm_U_GeoGLOWS_Oper_HRES,"+ CAP ∪ GeoGLOWS v2 Control HRES (0–7d, Q2yr)",RP=2,10885,7110,6420,65.319,58.980,87.661,35806,7071,19.748,30.327,1597.0,14.672,29.73,0.0
60,With_GS,2024_2025,CAP_Perm_U_DHM_0d,"+ CAP ∪ Google DHM (0d, Q2yr)",RP=2,10885,6983,6247,64.153,57.391,89.969,33624,6923,20.589,31.174,1470.0,13.505,27.36,0.0
61,With_GS,2024_2025,CAP_Perm_U_DHM_Oper,"+ CAP ∪ Google DHM (0–7d valid-date aligned, Q...",RP=2,10885,7152,6414,65.705,58.925,90.676,34061,6918,20.311,31.029,1639.0,15.057,30.51,0.0
62,With_GS,2024_2025,CAP_Perm_U_FFM,"+ CAP ∪ Google FFM (0–24h, τ=0.50)",tau=0.50,10885,8957,8443,82.288,77.565,92.013,41436,8213,19.821,31.947,3444.0,31.640,64.11,0.0


=== Table 2: Stream-Gauge Exceedance Precision vs. Disaster Catalogs ===


,rp_yr,gauge_n,gauge_precision_mean_pct,gauge_precision_median_pct,dhm_oper_without_gs_n_episodes,dhm_oper_without_gs_prec_pct,dhm_oper_without_gs_rec_pct,dhm_oper_with_gs_n_episodes,dhm_oper_with_gs_prec_pct,dhm_oper_with_gs_rec_pct,diff_gauge_vs_without_gs_pp,diff_gauge_vs_with_gs_pp
0,2,16971,52.411,56.410,30202,8.026,43.909,30202,28.631,34.726,44.39,23.78
1,5,16233,47.291,50.000,17522,9.582,27.621,17522,31.988,20.275,37.71,15.30
2,10,15041,44.097,50.000,11367,10.416,18.799,11367,32.911,13.241,33.68,11.19
3,20,12986,39.479,33.333,6889,10.756,11.876,6889,35.332,8.464,28.72,4.15
4,25,11894,38.750,33.333,5880,10.748,10.129,5880,35.969,7.340,28.00,2.78
5,50,7150,34.261,0.000,3529,11.193,6.362,3529,37.603,4.519,23.07,-3.34
6,100,3232,26.758,0.000,2018,12.091,4.038,2018,39.395,2.721,14.67,-12.64


=== Extended Data Table 3: Sensitivity Audit Matrix ===


,audit_type,catalog,lead_N,variant,n_events,tp_ev,recall_pct,n_episodes,tp_ep,precision_pct
0,Spatial_Attribution,Without_GS,7,perm_strict_GID1,5894,2277,38.633,65874,2509,3.809
1,Spatial_Attribution,Without_GS,7,perm_legacy_Admin0,5894,2859,48.507,4455,777,17.441
2,Spatial_Attribution,Without_GS,7,rest_strict_GID1,5894,423,7.177,12903,487,3.774
3,Spatial_Attribution,Without_GS,7,rest_legacy_Admin0,5894,840,14.252,867,248,28.604
4,Spatial_Attribution,Without_GS,15,perm_strict_GID1,5894,2452,41.602,66033,3365,5.096
...,...,...,...,...,...,...,...,...,...,...
169,Lead_Window_Sweep,With_GS,7,CAP_Perm_U_DHM_Oper_U_FFM050,27127,23122,85.236,108535,19714,18.164
170,Lead_Window_Sweep,With_GS,15,CAP_Perm_All,27127,13262,48.889,66033,16903,25.598
171,Lead_Window_Sweep,With_GS,15,DHM_Oper_RP2,27127,10915,40.237,31805,10419,32.759
172,Lead_Window_Sweep,With_GS,15,FFM_050,27127,20584,75.880,76226,24884,32.645


=== Extended Data Table 6a: Annual CAP Onboarding Trajectory (2021-2025) ===


,Year,New_Cat1_Flood_CAP,New_Cat1_ISOs,New_Cat2_NonFlood_CAP,Cumulative_Active_CAP_in_Window,Events_WoGS,CAP_Cov_WoGS,CAP_Cov_Pct_WoGS,Events_WGS,CAP_Cov_WGS,CAP_Cov_Pct_WGS
0,2021,45,"ARG,AUT,BEL,BGR,BIH,BRA,BRB,CAN,CHE,CHN,CZE,DE...",17,62,670,235,35.1,4110,1935,47.1
1,2022,12,"BEN,COG,CRI,GAB,GIN,JAM,LBY,MKD,PRT,SLB,TCD,TGO",16,90,974,362,37.2,4804,1961,40.8
2,2023,6,"BLR,BLZ,JOR,MDV,MOZ,NGA",4,100,1433,471,32.9,5769,2263,39.2
3,2024,4,"BDI,ETH,IRN,UKR",3,107,1781,720,40.4,6749,3085,45.7
4,2025,3,"AUS,PAN,PHL",5,115,1036,489,47.2,5695,3127,54.9


## 5. De-identified N = 400 CAP Keyword Validation Audit

In [6]:
val_path = paths.zenodo_dir / "04_validation_deidentified" / "validation_n400_deidentified.csv"
if val_path.exists():
    val_df = load_deidentified_validation_table(val_path)
    validate_deidentified_validation_table(val_df)
    cm = compute_confusion_matrix_table(val_df)
    sp = compute_spatial_precision_table(val_df)
    print("=== Spatial Pairing Precision (subset = all) ===")
    display(sp[sp["subset"] == "all"])
    print("=== Thematic Confusion Matrices (subset = all, v3 vs v4_reviewed) ===")
    display(cm[(cm["subset"] == "all") & (cm["label_version"].isin(["v3", "v4_reviewed"]))])
else:
    display(pd.read_csv(paths.results_dir / "ed_table4_validation_summary.csv"))

=== Spatial Pairing Precision (subset = all) ===


,label_version,subset,n_rows,spatial_yes,spatial_partial,spatial_no,precision_yes_pct,precision_yes_or_partial_pct,wald_halfwidth_pct,wald_fpc8508_halfwidth_pct,cp_lo,cp_hi
0,v1_initial200,all,200,23,3,174,11.50,13.00,4.4213,4.3693,7.4313,16.7547
6,v3,all,400,39,2,359,9.75,10.25,2.9070,2.8380,7.0254,13.0872
12,v4_reviewed,all,400,86,0,314,21.50,21.50,4.0260,3.9304,17.5727,25.8548
18,v4_v6_raw403,all,403,86,0,317,21.34,21.34,4.0001,3.9044,17.4389,25.6677


=== Thematic Confusion Matrices (subset = all, v3 vs v4_reviewed) ===


,label_version,subset,predictor,n_rows,n_predicted_positive,n_truth_positive,tp,fp,fn,tn,...,precision_wald_lo,precision_wald_hi,precision_wilson_lo,precision_wilson_hi,precision_cp_lo,precision_cp_hi,recall_wald_lo,recall_wald_hi,recall_cp_lo,recall_cp_hi
42,v3,all,exclusion_keep_all_sampled,400,400,206,206,194,0,0,...,46.6023,56.3977,46.6114,56.3601,46.4818,56.4958,100.0000,100.0000,98.2252,100.0000
43,v3,all,sheet_keyword_filter_status_Both,400,289,206,181,108,25,86,...,57.0521,68.2074,56.9206,68.0075,56.7723,68.2253,83.4049,92.3233,82.6080,91.9901
44,v3,all,filter02_not_event_excluded,400,330,206,206,124,0,70,...,57.1988,67.6497,57.0840,67.4785,56.9545,67.6688,100.0000,100.0000,98.2252,100.0000
45,v3,all,filter02_broad_flood_storm,400,235,206,197,38,9,156,...,79.1225,88.5371,78.5848,87.9866,78.4875,88.2957,92.8398,98.4223,91.8687,97.9831
46,v3,all,filter02_broad_flood_storm_with_sheet_translation,400,238,206,197,41,9,153,...,77.9757,87.5705,77.4650,87.0401,77.3642,87.3450,92.8398,98.4223,91.8687,97.9831
47,v3,all,filter02_explicit_flood,400,26,206,24,2,182,192,...,82.0651,102.5503,75.8584,97.8645,74.8697,99.0545,7.2693,16.0316,7.6091,16.8376
48,v3,all,annex_inclusion_keyword_any_text,400,224,206,191,33,15,161,...,80.6264,89.9093,80.0329,89.3136,79.9372,89.6370,89.1702,96.2667,88.2745,95.8675
84,v4_reviewed,all,exclusion_keep_all_sampled,400,400,220,220,180,0,0,...,50.1247,59.8753,50.1001,59.8048,49.9779,59.9475,100.0000,100.0000,98.3372,100.0000
85,v4_reviewed,all,sheet_keyword_filter_status_Both,400,289,220,190,99,30,81,...,60.2726,71.2153,60.0981,70.9767,59.9595,71.2017,81.8289,90.8984,81.1088,90.6068
86,v4_reviewed,all,filter02_not_event_excluded,400,330,220,220,110,0,70,...,61.5806,71.7528,61.4145,71.5353,61.2966,71.7344,100.0000,100.0000,98.3372,100.0000


## 6. Display Main and Extended Data Figures

In [7]:
import matplotlib.image as mpimg

figure_order = [
    "fig-1a.png",
    "fig-1b.png",
    "fig-2a.png",
    "fig-2b.png",
    "fig-2c.png",
    "fig-2_composite.png",
    "fig-3.png",
    "fig-4.png",
    "ed-fig-1.png",
    "ed-fig-2.png",
    "ed-fig-3.png",
    "ed-fig-4.png",
    "ed-fig-5.png",
]
inline_preview = os.environ.get("CAP_GAP_NB_INLINE_IMAGES", "0") == "1"
fig_rows = []
for fig_name in figure_order:
    fig_p = paths.figures_dir / fig_name
    assert fig_p.exists(), f"Missing expected figure: {fig_name}"
    arr = mpimg.imread(fig_p)
    fig_rows.append({
        "figure": fig_name,
        "width_px": int(arr.shape[1]),
        "height_px": int(arr.shape[0]),
        "size_kb": round(fig_p.stat().st_size / 1024.0, 1),
    })
    if inline_preview:
        display(Image(filename=str(fig_p), width=800))
display(pd.DataFrame(fig_rows))

,figure,width_px,height_px,size_kb
0,fig-1a.png,4474,1994,912.7
1,fig-1b.png,3067,1717,168.1
2,fig-2a.png,4414,1813,837.4
3,fig-2b.png,4414,1813,834.8
4,fig-2c.png,4414,1824,910.5
5,fig-2_composite.png,3796,5152,2227.3
6,fig-3.png,2235,2204,698.2
7,fig-4.png,3960,3390,1413.0
8,ed-fig-1.png,4440,1860,441.8
9,ed-fig-2.png,4320,3180,667.2
